# Step 1 · Collect web pages into a Volume

This notebook visits the websites you list, keeps only the **main readable content** of each page (no menus, ads or footers), and saves every page as a clean **Markdown (`.md`) file** in a Unity Catalog **Volume**.

The Volume is ready to use in two ways:
* **Genie Agent:** in your Genie Agent, open **Sources → Add**, pick the Volume, and turn on **content search**. Genie can then answer questions from these pages.
* **Vector Search:** run **`02_index_volume_to_vector_search`** to build a Vector Search index for custom AI agents. You can run it later and as often as you like.

## How to run
1. Attach the notebook to **Serverless** compute (environment version **6**).
2. Fill in the widgets at the top of the page. At minimum, set **Seed URLs**.
3. Click **Run all**. The log shows progress, and the last cells show a summary.

Running it again is safe. A page that was already collected is **overwritten** with its newest version, not duplicated.

## Settings (widgets)
| Widget | What it means |
|---|---|
| **Seed URLs** | Where to start. Put one or more web addresses separated by commas or new lines. A `sitemap.xml` address loads every page it lists. |
| **Catalog / Schema** | Where in Unity Catalog the Volumes live. They are created if missing. |
| **Content volume** | Holds only the clean `.md` pages. **Attach this one to Genie.** |
| **Staging volume** | Holds run logs, failed URLs and (optionally) raw HTML. Keep it separate so Genie only sees clean content. |
| **Max pages** | Stop after this many pages (safety limit). Genie content search supports up to 10,000 files per Volume. |
| **Follow links: depth** | How far to follow links, recursively. 0 means only the seed pages, 1 also follows the links found on them, 2 follows links on those pages, and so on. Combine it with **Max pages** and the URL patterns to control the crawl. |
| **Stay on same website** | Only follow links on the same website as the seed page. |
| **Include / Exclude URL pattern** | Optional regular expressions. For example, include `/help/` to keep only help pages. |
| **Parallel downloads** | How many pages to download at the same time. |
| **Delay per website (seconds)** | Pause between requests to the same website, so we're polite to it. |
| **Respect robots.txt** | Skip pages the website owner asks crawlers not to visit. |
| **Minimum words** | Pages with less readable text than this, such as login or empty pages, are skipped. |
| **Keep raw HTML** | Also save the original HTML to the staging Volume (useful for troubleshooting). |

The crawling code lives in `src/web_content_kit/crawler.py` in this project.

## 1. Install this project
This installs the project's code, `web_content_kit`, together with the libraries it needs for crawling: `requests`, and [`trafilatura`](https://trafilatura.readthedocs.io/), which extracts the main text of a page as Markdown.

In [ ]:
%uv pip install --quiet "..[crawl]"

In [ ]:
%restart_python

## 2. Settings
Change these values in the widget bar at the top of the notebook. The defaults are safe for a first test.

In [ ]:
dbutils.widgets.text("seed_urls", "https://docs.databricks.com/aws/en/ai-search/ai-search", "01 Seed URLs")
dbutils.widgets.text("catalog", "main", "02 Catalog")
dbutils.widgets.text("schema", "web_content", "03 Schema")
dbutils.widgets.text("volume", "web_content", "04 Content volume (attach to Genie)")
dbutils.widgets.text("staging_volume", "web_content_staging", "05 Staging volume (logs)")
dbutils.widgets.text("max_pages", "100", "06 Max pages")
dbutils.widgets.text("max_depth", "1", "07 Follow links: depth (0 = seed pages only)")
dbutils.widgets.dropdown("stay_on_domain", "true", ["true", "false"], "08 Stay on same website")
dbutils.widgets.text("include_url_pattern", "", "09 Include URL pattern (regex)")
dbutils.widgets.text("exclude_url_pattern", "", "10 Exclude URL pattern (regex)")
dbutils.widgets.text("concurrency", "16", "11 Parallel downloads")
dbutils.widgets.text("request_delay_seconds", "0.5", "12 Delay per website (seconds)")
dbutils.widgets.dropdown("respect_robots_txt", "true", ["true", "false"], "13 Respect robots.txt")
dbutils.widgets.text("min_words", "50", "14 Minimum words per page")
dbutils.widgets.dropdown("keep_raw_html", "false", ["true", "false"], "15 Keep raw HTML")
dbutils.widgets.text("user_agent", "DatabricksWebContentCollector/1.0", "16 User agent")

In [ ]:
from web_content_kit import settings as s
from web_content_kit.crawler import CrawlSettings
from web_content_kit.logs import get_logger

log = get_logger()
widget = dbutils.widgets.get

catalog = s.parse_identifier("Catalog", widget("catalog"))
schema = s.parse_identifier("Schema", widget("schema"))
volume = s.parse_identifier("Content volume", widget("volume"))
staging_volume = s.parse_identifier("Staging volume", widget("staging_volume"))
if volume == staging_volume:
    raise ValueError("The content volume and staging volume must be different, so Genie only sees clean pages.")

crawl_settings = CrawlSettings(
    seed_urls=s.parse_urls("Seed URLs", widget("seed_urls")),
    content_path=f"/Volumes/{catalog}/{schema}/{volume}",
    staging_path=f"/Volumes/{catalog}/{schema}/{staging_volume}",
    max_pages=s.parse_int("Max pages", widget("max_pages"), minimum=1),
    max_depth=s.parse_int("Max link depth", widget("max_depth"), minimum=0),
    stay_on_domain=s.parse_bool(widget("stay_on_domain")),
    include_pattern=s.parse_regex("Include URL pattern", widget("include_url_pattern")),
    exclude_pattern=s.parse_regex("Exclude URL pattern", widget("exclude_url_pattern")),
    concurrency=s.parse_int("Parallel downloads", widget("concurrency"), minimum=1),
    request_delay_seconds=s.parse_float("Delay per website", widget("request_delay_seconds")),
    respect_robots_txt=s.parse_bool(widget("respect_robots_txt")),
    min_words=s.parse_int("Minimum words", widget("min_words"), minimum=0),
    keep_raw_html=s.parse_bool(widget("keep_raw_html")),
    user_agent=widget("user_agent").strip() or "DatabricksWebContentCollector/1.0",
)
log.info("Seeds: %s", ", ".join(crawl_settings.seed_urls))
log.info("Content volume: %s | staging volume: %s", crawl_settings.content_path, crawl_settings.staging_path)
log.info(
    "Limits: max_pages=%d max_depth=%d concurrency=%d",
    crawl_settings.max_pages,
    crawl_settings.max_depth,
    crawl_settings.concurrency,
)

## 3. Create the schema and Volumes (if needed)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume}` "
    "COMMENT 'Clean Markdown copies of crawled web pages (one file per page). Attach to a Genie Agent as a source.'"
)
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{staging_volume}` "
    "COMMENT 'Crawl run logs, failed URLs and optional raw HTML for the web content volume.'"
)
log.info("Schema and volumes are ready.")

## 4. Crawl and save pages
Pages are downloaded in parallel. Each page is cleaned and written straight to the content Volume as it arrives. A page that fails (timeout, error, too little text) is logged and skipped, and it never stops the run. Afterwards, a record of every URL is saved to the staging Volume under `runs/<run id>/`.

In [ ]:
from web_content_kit.crawler import Crawler, new_run_id, write_run_log

run_id = new_run_id()
log.info("Starting crawl run %s", run_id)
outcomes = Crawler(crawl_settings, run_id).run()
run_dir = write_run_log(outcomes, crawl_settings.staging_path, run_id)

## 5. Summary

In [ ]:
from pathlib import Path

import pandas as pd

from web_content_kit.crawler import count_statuses

counts = count_statuses(outcomes)
total_files = sum(1 for _ in Path(crawl_settings.content_path).rglob("*.md"))
log.info(
    "Saved %d pages | duplicates %d | too short %d | skipped %d | failed %d",
    counts["saved"],
    counts["duplicate"],
    counts["too_short"],
    counts["skipped"],
    counts["failed"],
)
log.info("The content volume now holds %d Markdown files in total.", total_files)
if total_files > 10_000:
    log.warning("Genie content search supports up to 10,000 files per Volume. Consider lowering Max pages.")

displayHTML(f"""
<h3>Run {run_id} complete</h3>
<ul>
  <li><b>{counts["saved"]}</b> pages saved this run. <b>{total_files}</b> Markdown files in the volume.</li>
  <li>Content volume (attach to Genie): <code>{crawl_settings.content_path}</code></li>
  <li>Run log: <code>{run_dir}</code></li>
</ul>
<p><b>Next:</b> attach <code>{catalog}.{schema}.{volume}</code> to your Genie Agent
(<i>Sources → Add → Volume</i>, turn on content search), and/or run <code>02_index_volume_to_vector_search</code>.</p>
""")
results = pd.DataFrame(
    [
        {"url": o.url, "status": o.status, "title": o.title, "words": o.words, "detail": o.detail, "file": o.file_path}
        for o in outcomes
    ]
)
if not results.empty:
    display(results.sort_values(["status", "url"]))

### Preview one saved page
This is what Genie and the Vector Search step will read.

In [ ]:
first_saved = next((o.file_path for o in outcomes if o.status == "saved" and o.file_path), None)
if first_saved:
    print(Path(first_saved).read_text(encoding="utf-8")[:3000])
else:
    log.warning("No pages were saved. Check the seed URLs and the failures listed above.")